In [1]:

# Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path

# Display all columns when viewing datasets
pd.set_option('display.max_columns', None)

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

# Define the folder containing our original datasets

data_path = Path("../data/raw")

# Load the three CSV files

drug_reference = pd.read_csv(data_path / "drug_reference.csv")

patients = pd.read_csv(data_path / "patients.csv")

prescription_fills = pd.read_csv(data_path / "prescription_fills.csv")

print("All three datasets loaded successfully!")


All three datasets loaded successfully!


In [3]:

# Check the number of rows and columns

print("DRUG REFERENCE:", drug_reference.shape)

print("PATIENTS:", patients.shape)

print("PRESCRIPTION FILLS:", prescription_fills.shape)


DRUG REFERENCE: (14, 4)
PATIENTS: (2200, 7)
PRESCRIPTION FILLS: (25279, 10)


In [4]:

# Create a dictionary containing our three datasets

datasets = {
    "DRUG REFERENCE": drug_reference,
    "PATIENTS": patients,
    "PRESCRIPTION FILLS": prescription_fills
}

# Inspect columns and data types

for name, df in datasets.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("\nColumn Names:")
    print(df.columns.tolist())

    print("\nData Types and Non-Null Counts:")
    df.info()



DRUG REFERENCE

Column Names:
['drug_name', 'drug_class', 'measure', 'is_generic']

Data Types and Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   drug_name   14 non-null     object
 1   drug_class  14 non-null     object
 2   measure     14 non-null     object
 3   is_generic  14 non-null     object
dtypes: object(4)
memory usage: 576.0+ bytes

PATIENTS

Column Names:
['patient_id', 'birth_year', 'sex', 'state', 'plan_type', 'enrolled_from', 'enrolled_through']

Data Types and Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   patient_id        2200 non-null   object
 1   birth_year        2200 non-null   int64 
 2   sex               2200 non-null   object
 3   st

In [5]:

# Check missing values in all three datasets

for name, df in datasets.items():

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print(df.isnull().sum())



DRUG REFERENCE
drug_name     0
drug_class    0
measure       0
is_generic    0
dtype: int64

PATIENTS
patient_id          0
birth_year          0
sex                 0
state               0
plan_type           0
enrolled_from       0
enrolled_through    0
dtype: int64

PRESCRIPTION FILLS
fill_id         0
patient_id      0
filled_on       0
drug_name       0
days_supply     0
quantity        0
pharmacy_id     0
copay_usd       0
fill_channel    0
fill_status     0
dtype: int64


In [6]:

for name, df in datasets.items():

    print("\n", "=" * 50)
    print(name)
    print("=" * 50)

    found_empty = False

    for col in df.select_dtypes(include="object").columns:

        empty_count = df[col].map(
            lambda x: isinstance(x, str) and x.strip() == ""
        ).sum()

        if empty_count > 0:
            print(f"{col}: {empty_count} empty values")
            found_empty = True

    if not found_empty:
        print("No empty text values found.")



DRUG REFERENCE
No empty text values found.

PATIENTS
No empty text values found.

PRESCRIPTION FILLS
No empty text values found.


In [7]:

# Check duplicate records in all three datasets

for name, df in datasets.items():

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("Total duplicate rows:", df.duplicated().sum())



DRUG REFERENCE
Total duplicate rows: 0

PATIENTS
Total duplicate rows: 0

PRESCRIPTION FILLS
Total duplicate rows: 18


In [ ]:

# Find all rows involved in exact duplicates

duplicate_fills = prescription_fills[
    prescription_fills.duplicated(keep=False)
]

# Sort the duplicate records by fill_id

duplicate_fills = duplicate_fills.sort_values(
    by="fill_id"
)

print("Total rows involved in duplicate groups:",
      len(duplicate_fills))

display(duplicate_fills)


In [ ]:

# Check duplicate fill IDs

duplicate_fill_ids = prescription_fills[
    prescription_fills.duplicated(
        subset="fill_id",
        keep=False
    )
].sort_values(by="fill_id")

print("Rows with duplicate fill IDs:",
      len(duplicate_fill_ids))

display(duplicate_fill_ids)


In [10]:

# Check duplicate prescription IDs without displaying records

fill_id_counts = prescription_fills.groupby(
    "fill_id"
).size()

# Identify IDs appearing more than once
duplicate_id_counts = fill_id_counts[
    fill_id_counts > 1
]

# Count rows belonging to repeated IDs
total_repeated_id_rows = duplicate_id_counts.sum()

# Count extra occurrences beyond the first
extra_id_occurrences = (
    duplicate_id_counts - 1
).sum()

# Check whether repeated IDs have different information
repeated_records = prescription_fills[
    prescription_fills["fill_id"].isin(
        duplicate_id_counts.index
    )
]

other_columns = [
    col for col in prescription_fills.columns
    if col != "fill_id"
]

different_information = (
    repeated_records
    .groupby("fill_id")[other_columns]
    .nunique(dropna=False)
    .gt(1)
    .any(axis=1)
)

conflicting_ids = different_information.sum()

print("Repeated fill ID groups:", len(duplicate_id_counts))
print("Rows involved:", total_repeated_id_rows)
print("Extra occurrences:", extra_id_occurrences)
print("IDs with conflicting information:", conflicting_ids)
print(
    "IDs with identical information:",
    len(duplicate_id_counts) - conflicting_ids
)


Repeated fill ID groups: 18
Rows involved: 36
Extra occurrences: 18
IDs with conflicting information: 0
IDs with identical information: 18


In [1]:

# Create a working copy of prescription fills

prescription_fills_clean = prescription_fills.copy()

# Record the original row count
original_rows = len(prescription_fills_clean)

# Remove duplicate prescription IDs, keeping the first occurrence
prescription_fills_clean = prescription_fills_clean.drop_duplicates(
    subset="fill_id",
    keep="first"
).copy()

# Record the updated row count
cleaned_rows = len(prescription_fills_clean)

# Calculate removed records
removed_rows = original_rows - cleaned_rows

print("Original records:", original_rows)
print("Records after deduplication:", cleaned_rows)
print("Duplicate records removed:", removed_rows)

print(
    "Remaining duplicate fill IDs:",
    prescription_fills_clean["fill_id"].duplicated().sum()
)


NameError: name 'prescription_fills' is not defined

In [2]:
import pandas as pd
from pathlib import Path

# Locate the original dataset
data_path = Path("../data/raw")

# Load prescription fills again
prescription_fills = pd.read_csv(
    data_path / "prescription_fills.csv"
)

print("Dataset loaded successfully!")
print("Total records:", len(prescription_fills))

Dataset loaded successfully!
Total records: 25279


In [3]:
# Create a working copy
prescription_fills_clean = prescription_fills.copy()

original_rows = len(prescription_fills_clean)

# Remove duplicate fill IDs
prescription_fills_clean = prescription_fills_clean.drop_duplicates(
    subset="fill_id",
    keep="first"
).copy()

cleaned_rows = len(prescription_fills_clean)

print("Original records:", original_rows)
print("Records after deduplication:", cleaned_rows)
print("Duplicate records removed:", original_rows - cleaned_rows)

print(
    "Remaining duplicate fill IDs:",
    prescription_fills_clean["fill_id"].duplicated().sum()
)

Original records: 25279
Records after deduplication: 25261
Duplicate records removed: 18
Remaining duplicate fill IDs: 0


In [4]:

import pandas as pd
from pathlib import Path

# Dataset location
data_path = Path("../data/raw")

# Load original datasets
drug_reference = pd.read_csv(data_path / "drug_reference.csv")
patients = pd.read_csv(data_path / "patients.csv")
prescription_fills = pd.read_csv(data_path / "prescription_fills.csv")

# Create cleaned working copy
prescription_fills_clean = prescription_fills.drop_duplicates(
    subset="fill_id",
    keep="first"
).copy()

print("All datasets loaded successfully!")
print("Drug reference:", len(drug_reference))
print("Patients:", len(patients))
print("Prescription fills (clean):", len(prescription_fills_clean))


All datasets loaded successfully!
Drug reference: 14
Patients: 2200
Prescription fills (clean): 25261


In [5]:

# Define categorical columns to investigate

category_checks = {
    "DRUG REFERENCE": {
        "measure": drug_reference,
        "drug_class": drug_reference,
        "is_generic": drug_reference
    },
    "PATIENTS": {
        "sex": patients,
        "plan_type": patients
    },
    "PRESCRIPTION FILLS": {
        "fill_status": prescription_fills_clean,
        "fill_channel": prescription_fills_clean
    }
}

# Display category counts only

for dataset_name, columns in category_checks.items():

    print("\n" + "=" * 50)
    print(dataset_name)
    print("=" * 50)

    for col, df in columns.items():

        print(f"\n{col}")
        print("Unique categories:", df[col].nunique())

        print(df[col].value_counts(dropna=False))



DRUG REFERENCE

measure
Unique categories: 3
measure
diabetes               5
hypertension_ras       5
cholesterol_statins    4
Name: count, dtype: int64

drug_class
Unique categories: 7
drug_class
statins             4
arbs                3
sulfonylureas       2
ace_inhibitors      2
biguanides          1
dpp4_inhibitors     1
sglt2_inhibitors    1
Name: count, dtype: int64

is_generic
Unique categories: 2
is_generic
yes    12
no      2
Name: count, dtype: int64

PATIENTS

sex
Unique categories: 2
sex
F    1154
M    1046
Name: count, dtype: int64

plan_type
Unique categories: 2
plan_type
medicare_part_d    1600
commercial          600
Name: count, dtype: int64

PRESCRIPTION FILLS

fill_status
Unique categories: 2
fill_status
filled      24833
reversed      428
Name: count, dtype: int64

fill_channel
Unique categories: 2
fill_channel
retail        20561
mail_order     4700
Name: count, dtype: int64


In [6]:

# Count unique drug names in both datasets

reference_drug_count = drug_reference["drug_name"].nunique()

prescription_drug_count = prescription_fills_clean["drug_name"].nunique()

print("Unique drug names in reference:",
      reference_drug_count)

print("Unique drug names in prescriptions:",
      prescription_drug_count)


Unique drug names in reference: 14
Unique drug names in prescriptions: 42


In [7]:

# Standardize drug names temporarily for comparison

reference_names = (
    drug_reference["drug_name"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

prescription_names = (
    prescription_fills_clean["drug_name"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# Create sets of unique names
reference_set = set(reference_names.unique())
prescription_set = set(prescription_names.unique())

# Identify matching and unmatched names
matched_names = prescription_set.intersection(reference_set)

unmatched_names = prescription_set.difference(reference_set)

# Check prescription record counts
matched_rows = prescription_names.isin(reference_set).sum()

unmatched_rows = (~prescription_names.isin(reference_set)).sum()

# Display summary only — no drug names
print("Total unique prescription drug names:", len(prescription_set))
print("Matching unique drug names:", len(matched_names))
print("Unmatched unique drug names:", len(unmatched_names))

print("\nPrescription records matching reference:", matched_rows)
print("Prescription records not matching reference:", unmatched_rows)


Total unique prescription drug names: 14
Matching unique drug names: 14
Unmatched unique drug names: 0

Prescription records matching reference: 25261
Prescription records not matching reference: 0


In [8]:

# Compare original and standardized drug names

original_names = prescription_fills_clean["drug_name"].astype("string")

standardized_names = (
    original_names
    .str.strip()
    .str.casefold()
)

# Count original and standardized unique names
print("Original unique names:", original_names.nunique())

print("Standardized unique names:", standardized_names.nunique())

# Count records where standardization changed the text
changed_records = (
    original_names != standardized_names
).sum()

print("Records requiring text standardization:", changed_records)

# Verify reference uniqueness after standardization
reference_standardized = (
    drug_reference["drug_name"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

print(
    "Duplicate standardized reference names:",
    reference_standardized.duplicated().sum()
)


Original unique names: 42
Standardized unique names: 14
Records requiring text standardization: 3529
Duplicate standardized reference names: 0


In [9]:

# Convert date columns temporarily for inspection

fill_dates = pd.to_datetime(
    prescription_fills_clean["filled_on"],
    errors="coerce"
)

enrolled_from_dates = pd.to_datetime(
    patients["enrolled_from"],
    errors="coerce"
)

enrolled_through_dates = pd.to_datetime(
    patients["enrolled_through"],
    errors="coerce"
)

# Create date inspection summary

date_columns = {
    "Prescription fill dates": fill_dates,
    "Patient enrollment start": enrolled_from_dates,
    "Patient enrollment end": enrolled_through_dates
}

for name, dates in date_columns.items():

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("Total records:", len(dates))
    print("Valid dates:", dates.notna().sum())
    print("Invalid dates:", dates.isna().sum())
    print("Earliest date:", dates.min())
    print("Latest date:", dates.max())



Prescription fill dates
Total records: 25261
Valid dates: 25261
Invalid dates: 0
Earliest date: 2025-01-01 00:00:00
Latest date: 2025-12-31 00:00:00

Patient enrollment start
Total records: 2200
Valid dates: 2200
Invalid dates: 0
Earliest date: 2023-06-01 00:00:00
Latest date: 2025-04-06 00:00:00

Patient enrollment end
Total records: 2200
Valid dates: 2200
Invalid dates: 0
Earliest date: 2025-06-02 00:00:00
Latest date: 2025-12-31 00:00:00


In [10]:

# Check inconsistent enrollment periods

invalid_enrollment = (
    enrolled_through_dates < enrolled_from_dates
)

print(
    "Patients with invalid enrollment periods:",
    invalid_enrollment.sum()
)

# Check enrollment records with equal start and end dates

same_day_enrollment = (
    enrolled_through_dates == enrolled_from_dates
)

print(
    "Patients with same-day enrollment:",
    same_day_enrollment.sum()
)


Patients with invalid enrollment periods: 0
Patients with same-day enrollment: 0


In [11]:

# Extract prescription fill years

fill_years = fill_dates.dt.year

print("Prescription records by year:")

print(fill_years.value_counts().sort_index())

print("\nRecords within 2025:")

print((fill_years == 2025).sum())

print("\nRecords before 2025:")

print((fill_years < 2025).sum())

print("\nRecords after 2025:")

print((fill_years > 2025).sum())


Prescription records by year:
filled_on
2025    25261
Name: count, dtype: int64

Records within 2025:
25261

Records before 2025:
0

Records after 2025:
0


In [12]:

# Check duplicate patient IDs

duplicate_patient_ids = patients[
    patients["patient_id"].duplicated()
].shape[0]

# Check whether prescription patients exist in the patient dataset

orphan_fills = (
    ~prescription_fills_clean["patient_id"].isin(
        patients["patient_id"]
    )
).sum()

print("Duplicate patient ID rows:", duplicate_patient_ids)

print("Prescription records without matching patients:", orphan_fills)


Duplicate patient ID rows: 0
Prescription records without matching patients: 0


In [13]:

# Create temporary copies for date validation

patients_check = patients.copy()

fills_check = prescription_fills_clean.copy()

# Convert enrollment and prescription dates
patients_check["enrolled_from"] = pd.to_datetime(
    patients_check["enrolled_from"]
)

patients_check["enrolled_through"] = pd.to_datetime(
    patients_check["enrolled_through"]
)

fills_check["filled_on"] = pd.to_datetime(
    fills_check["filled_on"]
)

# Join prescription records with patient enrollment dates

merged_check = fills_check.merge(
    patients_check[
        ["patient_id", "enrolled_from", "enrolled_through"]
    ],
    on="patient_id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

# Identify records that matched a patient

matched = merged_check["_merge"].eq("both")

# Check prescription dates against enrollment periods

before_enrollment = (
    matched &
    (merged_check["filled_on"] < merged_check["enrolled_from"])
)

after_enrollment = (
    matched &
    (merged_check["filled_on"] > merged_check["enrolled_through"])
)

within_enrollment = (
    matched &
    (merged_check["filled_on"] >= merged_check["enrolled_from"]) &
    (merged_check["filled_on"] <= merged_check["enrolled_through"])
)

# Display aggregate results only

print("Total prescription records:", len(merged_check))

print("Matched patient records:", matched.sum())

print("Before enrollment:", before_enrollment.sum())

print("After enrollment:", after_enrollment.sum())

print("Within enrollment:", within_enrollment.sum())


Total prescription records: 25261
Matched patient records: 25261
Before enrollment: 0
After enrollment: 0
Within enrollment: 25261


In [14]:

# Inspect numerical columns

numeric_columns = ["days_supply", "quantity", "copay_usd"]

print("NUMERICAL DATA SUMMARY")
print("=" * 50)

print(
    prescription_fills_clean[numeric_columns].describe()
)


NUMERICAL DATA SUMMARY
        days_supply      quantity     copay_usd
count  25261.000000  25261.000000  25261.000000
mean      46.268952     46.523099     18.313239
std       25.363898     25.206137     12.313279
min        0.000000     30.000000      0.000000
25%       30.000000     30.000000      7.580000
50%       30.000000     30.000000     16.060000
75%       60.000000     60.000000     28.980000
max       90.000000     90.000000     42.000000


In [15]:

# Check numerical data quality

print("NUMERICAL QUALITY CHECK")
print("=" * 50)

print("\nDays Supply")
print("Zero values:", (prescription_fills_clean["days_supply"] == 0).sum())
print("Negative values:", (prescription_fills_clean["days_supply"] < 0).sum())
print("Above 365 days:", (prescription_fills_clean["days_supply"] > 365).sum())

print("\nQuantity")
print("Zero values:", (prescription_fills_clean["quantity"] == 0).sum())
print("Negative values:", (prescription_fills_clean["quantity"] < 0).sum())

print("\nCopay")
print("Zero values:", (prescription_fills_clean["copay_usd"] == 0).sum())
print("Negative values:", (prescription_fills_clean["copay_usd"] < 0).sum())


NUMERICAL QUALITY CHECK

Days Supply
Zero values: 134
Negative values: 0
Above 365 days: 0

Quantity
Zero values: 0
Negative values: 0

Copay
Zero values: 4
Negative values: 0


In [16]:

# Compare numerical values by prescription status

status_summary = prescription_fills_clean.groupby(
    "fill_status"
)[["days_supply", "quantity", "copay_usd"]].agg(
    ["count", "mean", "median", "min", "max"]
)

display(status_summary)


days_supply                           quantity                    \
                  count       mean median min max    count       mean median   
fill_status                                                                    
filled            24833  46.272702   30.0   0  90    24833  46.530021   30.0   
reversed            428  46.051402   30.0   0  90      428  46.121495   30.0   

                    copay_usd                                 
            min max     count       mean  median  min    max  
fill_status                                                   
filled       30  90     24833  18.310319  16.060  0.0  42.00  
reversed     30  90       428  18.482640  15.815  0.0  41.96

In [17]:

# Investigate zero days supply by prescription status

status_quality = prescription_fills_clean.groupby(
    "fill_status"
).agg(
    total_records=("fill_id", "count"),
    zero_days_supply=(
        "days_supply",
        lambda x: (x == 0).sum()
    ),
    positive_days_supply=(
        "days_supply",
        lambda x: (x > 0).sum()
    ),
    zero_copay=(
        "copay_usd",
        lambda x: (x == 0).sum()
    )
)

display(status_quality)


,total_records,zero_days_supply,positive_days_supply,zero_copay
fill_status,,,,
filled,24833,133,24700,3
reversed,428,1,427,1
